# Bucketing in Spark SQL with the Hive Catalog

A runnable customer lab: **four Parquet buckets without partitions**, then **four buckets within each country partition**. Metadata lives in the localhost Hive Metastore, data in HDFS, and execution on the localhost Spark standalone master/worker.

> Run in order in a fresh Python kernel inside WSL. Target: Apache Spark 3.5.x, V1 Parquet/ORC sources. All data is loaded with SQL `INSERT` statements. Outputs are intentionally unexecuted.

## Learning goals

- Calculate a customer's bucket and verify its physical file.
- Separate partition directories, logical buckets, physical files, and tasks.
- Interpret partition filters, bucket pruning, pushed filters, and data filters.
- Explain precisely what can be skipped and what ORC adds.

## 1. Start the course services

Open Windows Command Prompt and run `wsl`. In WSL, start services that are not already running:

```bash
start-dfs.sh
start-yarn.sh
mapred --daemon start historyserver
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
mkdir -p "$HOME/hive-logs"
nohup hive --service metastore > "$HOME/hive-logs/metastore.log" 2>&1 &
nohup hiveserver2 > "$HOME/hive-logs/hiveserver2.log" 2>&1 &
jps
ss -ltnp
hdfs getconf -confKey fs.defaultFS
hdfs dfsadmin -report
```

HDFS, YARN, Spark master/worker, and Hive Metastore should be up for the course. This notebook submits to **Spark standalone**, so YARN does not schedule its executors. HiveServer2 supports Beeline; Spark contacts the metastore directly on 9083.

Optional catalog inspection:

```bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER"
```

Start Jupyter in the WSL environment with the course PySpark installation:

```bash
jupyter lab --notebook-dir=/mnt/c/training
```

Adjust that mounted course path if necessary. The driver, worker, Hive and Hadoop must be in the **same WSL host** for these localhost addresses. Do not run the kernel in Windows or a separate container with these settings.

| Interface | URL | Inspect |
|---|---|---|
| Spark master | http://localhost:8080 | Registered worker and application |
| Spark application | Printed as `sc.uiWebUrl` below | SQL scans, stages, input bytes |
| ResourceManager | http://localhost:8088 | YARN health; this standalone job is not a YARN application |
| NodeManager | http://localhost:8042 | YARN node health |
| JobHistory | http://localhost:19888 | MapReduce history |
| NameNode | http://localhost:9870 | HDFS folders/files |

The course uses `hdfs://localhost:9000`. Change it below if `fs.defaultFS` reports another RPC port, such as 8020. Port 9870 is a web UI, not the HDFS data URI. WSL normally forwards web ports to Windows localhost.

## 2. Connect to the external Hive catalog

Restart the kernel if another Spark session already exists: `getOrCreate()` cannot move an existing context to another master. Use the compatible Hive client configuration installed for the course.

These are **Spark datasource tables registered in Hive**, created with `USING PARQUET`. They use Spark bucket hashing/file conventions, not Hive-native `STORED AS PARQUET` bucketing. Run the reads/writes through Spark SQL. A shared metastore does not guarantee Hive/Beeline bucket interoperability.

AQE is disabled and automatic bucket-scan selection is disabled to make teaching plans clear; bucketing itself stays enabled. These are lab settings, not universal production recommendations.

In [ ]:
import socket
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from IPython.core.magic import register_line_cell_magic

MASTER = "spark://localhost:7077"
HDFS = "hdfs://localhost:9000"
METASTORE = "thrift://localhost:9083"
for port in [7077, 9083, int(HDFS.rsplit(":", 1)[1])]:
    with socket.create_connection(("localhost", port), timeout=5):
        print(f"Reachable: localhost:{port}")

spark = (SparkSession.builder
    .appName("D375-Customer-Bucketing")
    .master(MASTER)
    .config("spark.hadoop.fs.defaultFS", HDFS)
    .config("spark.hadoop.hive.metastore.uris", METASTORE)
    .config("spark.sql.warehouse.dir", f"{HDFS}/user/hive/warehouse")
    .config("spark.sql.sources.useV1SourceList", "parquet,orc")
    .config("spark.sql.sources.bucketing.enabled", "true")
    .config("spark.sql.sources.bucketing.autoBucketedScan.enabled", "false")
    .config("spark.sql.parquet.filterPushdown", "true")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .enableHiveSupport()
    .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
assert sc.master == MASTER, "Restart the kernel: another Spark context is active."
assert spark.conf.get("spark.sql.catalogImplementation") == "hive"
assert sc._jsc.hadoopConfiguration().get("hive.metastore.uris") == METASTORE
print("Spark:", spark.version, "| Master:", sc.master)
print("HDFS:", sc._jsc.hadoopConfiguration().get("fs.defaultFS"))
print("Metastore:", sc._jsc.hadoopConfiguration().get("hive.metastore.uris"))
print("Application UI:", sc.uiWebUrl)
spark.sql("SHOW DATABASES").show(truncate=False)

@register_line_cell_magic
def sql(line, cell=None):
    """Execute one Spark SQL statement and display up to 100 rows."""
    statement = cell if cell is not None else line
    spark.sql(statement).show(100, truncate=False)

## 3. Isolate this lab run

The setup creates a new database each time, preserving existing course tables. Rerunning `INSERT OVERWRITE` resets only the lab table or explicitly named partition. Keep the printed database name for catalog/HDFS inspection. No external dataset or DataFrame writer is used.

In [ ]:
DB = "d375_bucketing_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
LAB_ROOT = f"{HDFS}/tmp/dataeng/{DB}.db"
spark.sql(f"CREATE DATABASE {DB} LOCATION '{LAB_ROOT}'")
spark.sql(f"USE {DB}")
print("Database:", DB, "| HDFS:", LAB_ROOT)
spark.sql(f"DESCRIBE DATABASE EXTENDED {DB}").show(truncate=False)

## 4. Example A: four Parquet buckets without partitions

`CLUSTERED BY (customer_id) INTO 4 BUCKETS` routes rows using the hash of `customer_id`. Without `PARTITIONED BY`, bucket files are directly inside the table location. `country` is an ordinary data column.

For these Spark-written tables the expression is `pmod(hash(customer_id), 4)`, producing bucket IDs 0-3. It is **not** `customer_id % 4`. Hashing depends on the type, so both tables declare the key `INT`. Different customer IDs can collide in the same bucket.

There is deliberately no `SORTED BY`. Hash bucketing does not arrange IDs into numeric ranges.

In [ ]:
%%sql
CREATE TABLE customers_b4 (
    customer_id INT,
    customer_name STRING,
    country STRING,
    segment STRING
)
USING PARQUET
CLUSTERED BY (customer_id) INTO 4 BUCKETS

In [ ]:
%%sql
INSERT OVERWRITE TABLE customers_b4 VALUES
    (1, 'Asha', 'IN', 'STANDARD'),
    (2, 'Bala', 'IN', 'VIP'),
    (3, 'Chen', 'US', 'STANDARD'),
    (4, 'Divya', 'IN', 'STANDARD'),
    (5, 'Ethan', 'US', 'VIP'),
    (6, 'Farah', 'IN', 'STANDARD'),
    (7, 'Grace', 'US', 'STANDARD'),
    (8, 'Hari', 'IN', 'VIP'),
    (9, 'Isha', 'US', 'STANDARD'),
    (10, 'Jai', 'IN', 'STANDARD'),
    (11, 'Kiran', 'US', 'VIP'),
    (12, 'Lina', 'IN', 'STANDARD'),
    (13, 'Maya', 'US', 'STANDARD'),
    (14, 'Noah', 'IN', 'VIP'),
    (15, 'Omar', 'US', 'STANDARD'),
    (16, 'Priya', 'IN', 'STANDARD')

In [ ]:
%%sql
DESCRIBE FORMATTED customers_b4

Look for `Provider: PARQUET`, `Num Buckets: 4`, `Bucket Columns: [customer_id]`, no partition columns, and an HDFS `Location`. Capture the file path at the scan and compare its bucket suffix to the computed hash.

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW customer_bucket_files AS
SELECT *, pmod(hash(customer_id), 4) AS expected_bucket,
       input_file_name() AS file_path,
       CAST(regexp_extract(input_file_name(), '_([0-9]{5})[.]', 1) AS INT) AS file_bucket
FROM customers_b4

In [ ]:
%%sql
SELECT customer_id, customer_name, country, expected_bucket, file_bucket,
       regexp_extract(file_path, '[^/]+$', 0) AS file_name
FROM customer_bucket_files
ORDER BY expected_bucket, customer_id

In [ ]:
%%sql
SELECT expected_bucket, sort_array(collect_set(customer_id)) AS customer_ids,
       count(*) AS rows_in_bucket, count(DISTINCT file_path) AS physical_files
FROM customer_bucket_files
GROUP BY expected_bucket ORDER BY expected_bucket

In [ ]:
assert spark.table("customers_b4").count() == 16
assert spark.sql("""SELECT * FROM customer_bucket_files
    WHERE file_bucket IS NULL OR file_bucket <> expected_bucket""").count() == 0
assert spark.sql("SELECT DISTINCT expected_bucket FROM customer_bucket_files").count() == 4
print("Verified: 16 rows, four occupied buckets, hash matches file bucket IDs.")

# Read-only HDFS inventory via Hadoop; includes files with no matching query rows.
def list_table_files(table):
    details = spark.sql(f"DESCRIBE FORMATTED {table}").collect()
    location = next(r.data_type for r in details if r.col_name.strip() == "Location")
    path = sc._jvm.org.apache.hadoop.fs.Path(location)
    fs = path.getFileSystem(sc._jsc.hadoopConfiguration())
    files = fs.listFiles(path, True)
    print(location)
    while files.hasNext():
        item = files.next()
        if item.getPath().getName().endswith((".parquet", ".orc")):
            print(item.getLen(), item.getPath().toString())

list_table_files("customers_b4")

### Four buckets does not promise four files

In a typical Spark name `part-00000-<uuid>_00003.c000.snappy.parquet`, `_00003` identifies **bucket 3**. The first `part-00000` identifies a writer task, not the bucket. Names and UUIDs vary.

Multiple writer tasks, repeated inserts and file rolling can produce several files for one bucket. Empty buckets need not have files. The invariant is correct row placement, not exactly four files. HDFS blocks and Spark execution partitions are separate concepts.

Bucket pruning retains **all files** for each selected bucket, including files from earlier insert batches.

## 5. Example B: bucketing within country partitions

This example represents customer membership by country. The same 16 IDs are deliberately inserted into both countries to make routing visible. The example's unique key is `(country, customer_id)`; it is not a model of one home country per person.

For each row:

1. `country` selects a partition directory.
2. `pmod(hash(customer_id), 4)` selects bucket 0-3 **inside that directory**.
3. Spark writes a Parquet file carrying that bucket ID.

Two country directories have eight possible **(country, bucket ID)** combinations. Bucket IDs start at zero in each directory, not 4-7 in the second country. The same typed ID maps to the same bucket ID in both countries.

In [ ]:
%%sql
CREATE TABLE customers_country_b4 (
    customer_id INT,
    customer_name STRING,
    segment STRING,
    country STRING
)
USING PARQUET
PARTITIONED BY (country)
CLUSTERED BY (customer_id) INTO 4 BUCKETS

In [ ]:
%%sql
INSERT OVERWRITE TABLE customers_country_b4 PARTITION (country='IN')
SELECT customer_id, customer_name, segment FROM customers_b4

In [ ]:
%%sql
INSERT OVERWRITE TABLE customers_country_b4 PARTITION (country='US')
SELECT customer_id, customer_name, segment FROM customers_b4

In [ ]:
%%sql
SHOW PARTITIONS customers_country_b4

In [ ]:
%%sql
DESCRIBE FORMATTED customers_country_b4

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW country_bucket_files AS
SELECT *, pmod(hash(customer_id), 4) AS expected_bucket,
       input_file_name() AS file_path,
       CAST(regexp_extract(input_file_name(), '_([0-9]{5})[.]', 1) AS INT) AS file_bucket
FROM customers_country_b4

In [ ]:
%%sql
SELECT country, expected_bucket,
       sort_array(collect_set(customer_id)) AS customer_ids,
       count(*) AS row_count, count(DISTINCT file_path) AS physical_files
FROM country_bucket_files
GROUP BY country, expected_bucket ORDER BY country, expected_bucket

In [ ]:
assert spark.table("customers_country_b4").count() == 32
assert spark.sql("""SELECT * FROM country_bucket_files
    WHERE file_bucket IS NULL OR file_bucket <> expected_bucket""").count() == 0
assert spark.sql("""SELECT DISTINCT country, expected_bucket
    FROM country_bucket_files""").count() == 8
list_table_files("customers_country_b4")

Conceptual layout (bucket identities, not guaranteed file counts):

```text
customers_b4/                     customers_country_b4/
  ..._00000...parquet               country=IN/
  ..._00001...parquet                 ..._00000...parquet through ..._00003...parquet
  ..._00002...parquet               country=US/
  ..._00003...parquet                 ..._00000...parquet through ..._00003...parquet
```

There are no `bucket=0/` directories. Spark normally reconstructs the partition column `country` from directory/catalog values instead of storing it in the Parquet payload.

## 6. Understand the scan-plan fields

| Field | Meaning | What it can avoid |
|---|---|---|
| `PartitionFilters` | Conditions on directory partition keys | Nonmatching country directories |
| `SelectedBucketsCount` | Bucket IDs chosen using catalog metadata and eligible key predicates | Files belonging to other bucket IDs |
| `DataFilters` | Conditions involving stored data columns; may also inform bucket selection | Not evidence of skipping by itself |
| `PushedFilters` | Predicates translated for the file reader | Eligible row groups, if reader settings/statistics allow |
| `ReadSchema` | Physical columns needed for projection and predicates | Unneeded column chunks |
| `Filter` above scan | Exact evaluation on candidate rows | Removes rows after reading; not file skipping |

These roles overlap. `customer_id=5` can be a data filter, a pushed filter, and the reason for selecting one bucket. A partition column can appear in the output without appearing in `ReadSchema`.

`PushedFilters` does not prove bytes were skipped: statistics can overlap the predicate. Bucket collisions and coarse statistics mean an exact residual `Filter` is still normal.

In [ ]:
def inspect_sql(statement):
    print(statement)
    query = spark.sql(statement)
    query.explain(mode="formatted")
    query.show(100, truncate=False)

# Retain the bucket key in the projection to make bucketed scan output visible.
inspect_sql("SELECT customer_id, customer_name FROM customers_b4 WHERE customer_id = 5")

Expect no partition filter and `SelectedBucketsCount: 1 out of 4`. The literal hashes as the `INT` key. Spark retains files for that bucket, then still checks customer IDs. Compute the mapping for the next `IN` example rather than assuming that three values require three distinct buckets.

In [ ]:
%%sql
SELECT customer_id, pmod(hash(customer_id), 4) AS bucket_id
FROM customers_b4 WHERE customer_id IN (5, 6, 7)
ORDER BY customer_id

In [ ]:
cases = [
    ("No filter: both partitions, all buckets", "TRUE"),
    ("Partition only: IN, all buckets", "country = 'IN'"),
    ("Equality only: one bucket ID in BOTH countries", "customer_id = 5"),
    ("Partition + equality: one bucket ID in IN", "country = 'IN' AND customer_id = 5"),
    ("IN list: union of bucket IDs in IN", "country = 'IN' AND customer_id IN (5, 6, 7)"),
    ("Range: IN, all hash buckets", "country = 'IN' AND customer_id BETWEEN 5 AND 7"),
    ("Non-key filter: IN, all buckets", "country = 'IN' AND segment = 'VIP'"),
    ("Expression hides key equality", "country = 'IN' AND abs(customer_id) = 5"),
    ("OR: VIP rows could be in ANY bucket", "country = 'IN' AND (customer_id = 5 OR segment = 'VIP')"),
]
for label, predicate in cases:
    print("\n", label)
    inspect_sql(f"SELECT customer_id, customer_name, country, segment "
                f"FROM customers_country_b4 WHERE {predicate}")

### Predict which locations survive

Let `k` be the number of distinct bucket IDs for customers 5, 6, 7.

| Predicate on partitioned table | Country partitions retained | (country, bucket) combinations retained |
|---|---:|---:|
| None | 2 | 8 |
| `country='IN'` | 1 | 4 |
| `customer_id=5` | 2 | 2 |
| `country='IN' AND customer_id=5` | 1 | 1 |
| `country='IN' AND customer_id IN (5,6,7)` | 1 | k |
| `country='IN' AND customer_id BETWEEN 5 AND 7` | 1 | 4 |
| `country='IN' AND segment='VIP'` | 1 | 4 |

`SelectedBucketsCount: 1 out of 4` counts **bucket IDs**, not physical files or partition/bucket pairs. Without a country predicate, that one bucket ID must be considered in both countries.

Ranges do not map to adjacent hash buckets. Expressions may hide equality unless Catalyst simplifies them. With `AND`, a usable equality can still narrow buckets alongside another condition. With `OR`, an unrestricted branch makes the bucket union unrestricted.

In the unpartitioned table, `country='IN'` is a data filter, not a partition filter:

In [ ]:
inspect_sql("SELECT customer_id, customer_name FROM customers_b4 WHERE country = 'IN'")

## 7. Separate bucketing from Parquet pushdown

Run the same query three ways, creating a fresh query after each setting change:

- Both enabled: one country directory and one bucket ID are candidates; row-group skipping may help too.
- Bucketing disabled: country pruning remains, but all bucket files in IN are candidates. Parquet pushdown is still enabled.
- Parquet pushdown disabled: partition and bucket pruning remain, while Parquet predicate-based row-group elimination is disabled.

In Spark 3.5, the plan can still list `PushedFilters` when `spark.sql.parquet.filterPushdown=false`. The plan displays translated predicates; the reader setting controls whether Parquet uses them. Check the setting too.

In [ ]:
probe = """SELECT customer_id, customer_name, segment FROM customers_country_b4
           WHERE country = 'IN' AND customer_id = 5 AND segment = 'VIP'"""
keys = ["spark.sql.sources.bucketing.enabled", "spark.sql.parquet.filterPushdown"]
saved = {key: spark.conf.get(key) for key in keys}
results = []
try:
    for bucketing, pushdown in [("true", "true"), ("false", "true"), ("true", "false")]:
        spark.conf.set(keys[0], bucketing)
        spark.conf.set(keys[1], pushdown)
        print(f"\nBucketing={bucketing}; Parquet filterPushdown={pushdown}")
        query = spark.sql(probe)
        query.explain("formatted")
        results.append(query.collect())  # Execute this exact tiny query, without a LIMIT.
        print(results[-1])
finally:
    for key, value in saved.items():
        spark.conf.set(key, value)
assert results[0] == results[1] == results[2]
assert len(results[0]) == 1 and results[0][0].customer_id == 5

## 8. Exactly how data skipping works

For `country='IN' AND customer_id=5 AND segment='VIP'`:

1. **Partition pruning:** Spark uses catalog/directory partition information to exclude US before scanning its data files. Metadata/listing work may still occur.
2. **Bucket pruning:** Spark calculates the bucket for `CAST(5 AS INT)` and retains all files for that bucket within IN. It need not inspect row values in excluded bucket files.
3. **Column pruning:** the reader requests columns needed by the projection and predicates. Selecting only the name still requires ID and segment to evaluate the filter.
4. **Reader filtering:** Parquet statistics may rule out row groups inside candidate files. A group with minimum ID 20 cannot contain 5. A min/max range of 1-100 cannot establish whether 5 exists.
5. **Exact filtering:** surviving data is decoded and predicates checked. Hash collisions and overlapping statistics explain the residual filter.

If all row groups are excluded, the file's data pages may be avoided, but opening the file and reading its footer still costs I/O. That is different from excluding an entire file through partition/bucket metadata.

### Illustrative statistics, not measured output

| Group in a retained bucket file | customer_id min/max | Can equality to 5 rule it out? |
|---|---|---|
| A | 1-100 | No |
| B | 101-200 | Yes |
| C | 201-300 | Yes |

Bucketing does not itself create those ranges. Optional `SORTED BY (customer_id)` can improve ranges within each output file if there are enough rows for multiple groups. It does not change hash routing or globally order files from different inserts.

This tiny dataset demonstrates routing, **not speed**. A bucket file may have only one row group. Choosing one of four buckets does not promise 75% fewer bytes: sizes, skew, compression, file counts and metadata overhead differ.

## 9. ORC: what can be better, and why

**ORC does not improve the bucket hash.** Bucketing chooses a layout; Parquet and ORC store rows within it. ORC can make scanning the retained bucket files more selective:

- File/stripe statistics and row-group indexes, including seek positions, allow irrelevant regions to be skipped.
- Optional column Bloom filters can rule out equality/`IN` values even when min/max ranges overlap. False positives remain possible, so exact filtering is still required.
- Hive has mature ORC reader/vectorization integration. Spark supports vectorized readers for both ORC and Parquet, so vectorization is not unique to ORC.

Parquet also supports row-group statistics and skipping; format-level Bloom filters/page indexes exist, but deployed writer/reader support must be checked. Actual results depend on data, sorting, indexes, compression, and engine settings. ORC is not universally faster.

See [ORC indexes](https://orc.apache.org/docs/indexes.html), [Spark ORC options](https://spark.apache.org/docs/3.5.7/sql-data-sources-orc.html), and [Spark Parquet options](https://spark.apache.org/docs/3.5.7/sql-data-sources-parquet.html).

The course's `mapreduce.job.user.classpath.first` property is set and viewed before the ORC example. It controls MapReduce classpath precedence, **not bucketing or skipping**. It does not reorder an already-started Spark executor's classpath. For a Hive job, issue the setting separately in its Beeline session.

In [ ]:
%%sql
SET mapreduce.job.user.classpath.first=true

In [ ]:
%%sql
SET mapreduce.job.user.classpath.first

In [ ]:
%%sql
CREATE TABLE customers_b4_orc (
    customer_id INT,
    customer_name STRING,
    country STRING,
    segment STRING
)
USING ORC
OPTIONS ('orc.bloom.filter.columns'='customer_id')
CLUSTERED BY (customer_id) INTO 4 BUCKETS

In [ ]:
%%sql
INSERT OVERWRITE TABLE customers_b4_orc
SELECT customer_id, customer_name, country, segment FROM customers_b4

In [ ]:
inspect_sql("SELECT customer_id, customer_name FROM customers_b4_orc WHERE customer_id = 5")
assert spark.table("customers_b4_orc").count() == 16
list_table_files("customers_b4_orc")

Both formats should select the same logical bucket for the same typed key and Spark bucket count. The ORC plan does not prove that a Bloom filter was consulted or that any row groups were eliminated. Use ORC inspection tools/reader metrics with larger files for that evidence; sixteen rows cannot establish a format performance advantage.

## 10. Runtime evidence and common traps

Use the application's SQL tab (`sc.uiWebUrl`), not just the master summary on 8080. Compare section 7 executions using available scan file/byte metrics, stage input bytes, records, and duration. Metric accounting varies; file metrics can count candidates rather than files returning rows.

- `explain` does not execute a query. Section 7 uses `collect()` on the exact tiny query.
- `input_file_name()` on returned rows identifies contributing files, not every file opened.
- `DataFrame.inputFiles()` is a best-effort inventory, not proof of predicate-specific pruning.
- Task count is not bucket-pruning evidence: bucketed scans can retain empty task partitions.
- Cache and warm filesystem effects distort timings; these tables are not cached.
- Reading a raw Parquet path loses the catalog bucket specification. Query the registered table for bucket-aware planning.

Bucketing can also help compatible joins avoid shuffles when the planner uses the metadata. That is a separate benefit from filtering: an unfiltered scan is not made smaller just by bucketing.

## 11. Check your understanding

1. Why is a row filter still needed after `customer_id=5` selects one bucket?
2. With 12 country partitions, how many partition/bucket pairs remain for ID 5 without a country predicate?
3. Why do ranges not narrow hash buckets?
4. Does `PushedFilters` prove a row group was skipped?
5. Does another insert change the declared bucket count?

**Answers:** (1) Other IDs collide in that bucket. (2) Twelve, one per country. (3) Hashing does not preserve numeric order. (4) No; statistics and reader settings determine actual skipping. (5) No; an insert may add files to existing buckets.

## References and version scope

- [Spark 3.5.7 datasource table SQL](https://spark.apache.org/docs/3.5.7/sql-ref-syntax-ddl-create-table-datasource.html): partition and bucket DDL.
- [Spark bucketing API documentation](https://spark.apache.org/docs/3.5.7/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.bucketBy.html): Spark and Hive hashing differ. The lab uses SQL instead of this writer API.
- [Spark 3.5.7 scan planner](https://github.com/apache/spark/blob/v3.5.7/sql/core/src/main/scala/org/apache/spark/sql/execution/datasources/FileSourceStrategy.scala): single-column bucket predicates and separation of partition, bucket and data filters.

The examples use one bucket column and V1 file sources. Other versions/providers can show different plans. If `SelectedBucketsCount` is missing, inspect table metadata, reader/provider, projected bucket key, and both bucketing settings before claiming pruning occurred.